# Module 1: Anomaly prompt construction

Produces one anomaly prompt per defect-free image. No image is generated here. The prompts are
handed to an image generation model in the next step, and the resulting anomaly images are the
input to Module 2.

## Requirements

Kaggle notebook, accelerator set to GPU T4 x2, internet off.

Attach two mounts:

1. `Qwen/Qwen2.5-VL-7B-Instruct`, as a Model.
2. MVTec AD 2, as a Dataset.

## Steps

1. Open section 2 and set `N_IMAGES`, the categories you want, and the output directory. If the
   mounts are named differently on your kernel, set `MODEL_ID` and `DATA_ROOT_ID` as well.
2. Run all cells in order.
3. Read the prompts in section 9 before spending any generation budget. Discard any that describe
   a defect the image cannot support.
4. Pass each prompt together with its normal image to the image generation model, and save the
   returned anomaly image for Module 2.

## Output

```
<output dir>/
  <category>/Normal_<category>_<id>.png    the sampled normal images
  prompts.csv                              one row per image: tag, family, defect, prompt, seconds
  manifest.json                            the same content as JSON
```

Each image gets one prompt. Prompts within a category are forced apart: every image is assigned a
different defect family from a fixed vocabulary before the model sees it, the size word is sampled
per call, and a repeated defect name is retried.

## 1. Environment

Loads the libraries and prints the versions actually in use.

Weights and data come from the attached mounts, so nothing downloads and the kernel can run with
internet off. The model is loaded in fp16 rather than 4-bit, which avoids a `bitsandbytes`
install. T4 is Turing, so bf16 and flash-attention are unavailable and the dtype and attention
implementation are set explicitly.

In [ ]:
import torch, transformers, os, glob, json, re, time, random, shutil, csv

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)

n_gpu = torch.cuda.device_count()
print("cuda        ", n_gpu, "x", torch.cuda.get_device_name(0) if n_gpu else "cpu")
if n_gpu:
    tot = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n_gpu)) / 1e9
    print(f"vram         {tot:.0f} GB total across {n_gpu} device(s)")
# torch may report True on T4 (emulated), but Turing has no NATIVE bf16 -- emulated
# bf16 is slower than fp16 here, so the model is loaded in float16 regardless.
print("bf16 (may be emulated):", torch.cuda.is_bf16_supported() if n_gpu else False)

## 2. Config

The only cell you need to edit. Nothing tunable is hidden in the cells below.

| block | contains |
|---|---|
| run knobs | `SEED`, mount paths, `LOAD_4BIT`, pixel caps, `N_IMAGES`, `DEFECT_SOURCE`, output dir |
| dataset context | one paragraph telling the model what kind of images these are |
| per-category `CONFIG` | `object`, `families`, `defects`, `notes`, optional `max_pixels` |
| prompt text | `FAMILIES`, `SIZE_WORDS`, `TEMPLATE`, `HARD_RULES` |
| model wording | `SYS_VLM1` and `ASK_VLM1` |

The per-category entries are hints rather than rules. `defects` seeds the vocabulary and `notes`
warns about scenes that are commonly misread, but the model still looks at the image.

To add a category, add an entry to `CONFIG` and list it in `CATS`.

In [ ]:
# ---------------------------------------------------------------- run knobs
SEED          = 0
# Mounted paths. Each is used if it exists; otherwise the notebook falls back to searching
# /kaggle/input, so a re-mount under a different slug does not break the run.
MODEL_ID      = "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/7b-instruct/2"
DATA_ROOT_ID  = "/kaggle/input/datasets/abhaykdas/mvtec-ad2-reducded"
LOAD_4BIT     = False        # False -> fp16 split across both T4s (no internet needed)
MAX_PIXELS    = 1280 * 28 * 28   # visual-token cap: AD2 natives are far bigger than needed
MIN_PIXELS    = 256 * 28 * 28
OUT_DIR       = "/kaggle/working/Local Testing"

DEFECT_SOURCE = "config"     # config = VLM must pick from CONFIG[cat]["defects"]
                             # derive = it may name its own, using the list as examples
N_IMAGES      = 3            # normal images sampled per category; ONE prompt per image

# Shown to the VLM so it knows what kind of image it is looking at. Dataset-level, not
# category-level -- edit this once per dataset.
DATASET_CONTEXT = (
    "High-resolution industrial inspection images from a fixed camera rig under controlled "
    "illumination. Every image in the set shows the same object class under the same "
    "acquisition setting. Real defects in this kind of data are small relative to the frame, "
    "occur anywhere across the full height and width including right at the image borders, "
    "and are often low contrast against the material they sit on."
)

# ---------------------------------------------------------------- per-category config
#   defects : allowed defect list, used only when DEFECT_SOURCE == "config"
#   notes   : category hazards, always passed to the VLM as a hint (e.g. "air bubbles are
#             normal, not defects" stops it proposing bubbles as an anomaly)
# Both fields are optional. Delete a category to skip it; add one to extend to a new dataset.
CONFIG = {
    "can":         {"object": "a printed aluminium soda can lying on a plain background",
                    "families": ["print or coating defect", "scratch or cut", "stain or discolouration",
                                 "deformation"],
                    "defects": ["scratch tearing through the printed label so the bare silver metal underneath shows",
                                "smeared print",
                                "faded patch in the print",
                                "dark ink speck",
                                "dented rim"],
                    "notes": 
"diffuse bright-field front light on a printed soda can. The can rotates between "
                             "shots so print position varies, and the metal throws bright specular highlights -- "
                             "both normal. Real defects here are tiny against busy print"},
    "fabric":      {"object": "a flat sheet of printed woven fabric filling the frame",
                    "families": ["hole or tear", "scratch or cut", "fibre or thread contamination",
                                 "stain or discolouration", "foreign body contamination"],
                    "defects": ["tiny red stain spot",
                                "leftover thread lying on the surface",
                                "loose scrap of fabric on the surface",
                                "hole through the weave",
                                "cut across the weave",
                                "pulled thread"],
                    "notes": 
"diffuse bright-field front light for fabric inspection. The printed pattern itself "
                             "varies between samples, so pattern variation is normal. Real defects are tiny and "
                             "low-contrast against that print"},
    "fruit_jelly": {"object": "a sealed tub of solid set fruit jelly with pieces of fruit suspended inside it",
                    "families": ["foreign body contamination", "fibre or thread contamination",
                                 "stain or discolouration", "mould or rot"],
                    "defects": ["glass shard pressed into the surface",
                                "clear plastic fragment",
                                "dark speck of debris",
                                "small insect on the surface",
                                "patch of mould growing on the jelly",
                                "rotten discoloured fruit piece"],
                    "notes": 
"diffuse bright-field back and front light through semi-transparent jelly. The "
                             "amount, size and layout of the fruit pieces vary between samples and are normal. "
                             "Real defects are low-contrast and partly see-through"},
    "rice":        {"object": "loose white rice grains in bulk, filling the frame",
                    "families": ["foreign body contamination", "fibre or thread contamination",
                                 "stain or discolouration"],
                    "defects": ["clear plastic pellet among the grains",
                                "small stone among the grains",
                                "small piece of glass among the grains",
                                "dark grit particle",
                                "stray fibre across the grains"],
                    "notes": 
"diffuse bright-field front light on rice grains in bulk. Grain layout is random and "
                             "normal. The hard case is contamination that is itself (semi-)transparent, so real "
                             "defects are very low contrast"},
    "sheet_metal": {"object": "several flat strips of machined sheet metal",
                    "families": ["hole or tear", "scratch or cut", "foreign body contamination",
                                 "stain or discolouration"],
                    "defects": ["hole punched through the metal",
                                "deep scratch",
                                "cut in the edge",
                                "short piece of metal wire lying across the surface",
                                "curl of metal swarf on the surface",
                                "dark speck of grit"],
                    "notes": 
"directed dark-field front light on strips of sheet metal. Random specular "
                             "reflections look like defects but are normal. The frame is a wide strip and real "
                             "samples can carry several defects of different sizes",
                    "max_pixels": 2560 * 28 * 28},
    "vial":        {"object": "a single sealed glass vial of clear sparkling liquid with a printed QR label",
                    "families": ["foreign body contamination", "hole or tear",
                                 "print or coating defect", "missing or broken part"],
                    "defects": ["dark particle suspended in the liquid",
                                "small insect trapped inside the vial",
                                "torn corner of the QR label",
                                "scratched-through print on the label",
                                "chip in the glass rim"],
                    "notes": 
"diffuse bright-field back light through a transparent vial of clear sparkling "
                             "liquid. Air bubbles, vial rotation, fill level and QR-code position all vary and are "
                             "normal, not defects. Real defects are small, low-contrast and often seen through "
                             "glass"},
    "wallplugs":   {"object": "a loose pile of plastic wall plugs on a plain background",
                    "families": ["missing or broken part", "crack or split", "deformation",
                                 "scratch or cut", "foreign body contamination"],
                    "defects": ["crack running down the shaft",
                                "snapped-off tip",
                                "missing flared collar",
                                "deformed misshapen plug",
                                "plastic burr left by the mould",
                                "deep scratch along the body"],
                    "notes": 
"diffuse bright-field front light on a loose pile of plastic wall plugs. They touch, "
                             "overlap and get cut off by the image frame, and their number and placement vary -- "
                             "all normal. Real defects sit on one individual plug"},
    "walnuts":     {"object": "several whole walnuts in their shells on a plain background",
                    "families": ["crack or split", "hole or tear", "missing or broken part",
                                 "mould or rot", "foreign body contamination"],
                    "defects": ["broken shell showing the kernel inside",
                                "crack splitting the shell open",
                                "hole bored through the shell exposing the kernel",
                                "patch of mould on the shell",
                                "dark speck of grit"],
                    "notes": 
"diffuse bright-field front light on loose walnuts. They touch, overlap and get cut "
                             "off by the image frame, and vary widely in size, shape and shell structure. The "
                             "natural shell grain is normal and is not a crack"},
}

# Optional: pin which families a category draws, in order, e.g.
#   FAMILY_PLAN = {"rice": ["foreign contamination", "stain or discolouration", "foreign object"]}
# Empty -> families are sampled per category using SEED.
FAMILY_PLAN = {}

# ---------------------------------------------------------------- prompt text (editable)
# Defect families: the diversity axis. Each image in a category is assigned a DIFFERENT
# family, so the images of one category cannot collapse onto the same defect.
#
# Grounded in MVTec AD 2 Table 3 ("Description of Occurring Defects"), which lists, across the
# eight categories: print defects, scratches, cuts, holes, colour inconsistencies, loose
# threads and extra fabric pieces, foreign object contamination of varying texture and size,
# (semi-)transparent plastic contamination, cracks, missing parts, broken pieces, and damaged
# or missing QR codes.
#
# The hints name CONCRETE industrial materials on purpose. Left open, a VLM proposes things
# like leaves or insects, which never occur on a sealed inspection rig -- what actually
# contaminates these lines is glass, plastic, metal swarf, grit, and fibres.
FAMILIES = {
    "scratch or cut":        "a thin abraded or incised line that breaks the surface finish "
                             "and exposes the material under it",
    "hole or tear":          "material missing right through, with visibly torn or cut edges",
    "crack or split":        "a fracture line in a rigid material, opening slightly along its length",
    "missing or broken part": "a piece of the item snapped off or absent, leaving a blunt "
                             "broken face",
    "foreign body contamination": "one small thing that does not belong -- a glass shard, a "
                             "plastic sliver, a metal shaving, a piece of grit, or on food "
                             "and liquid products a small insect",
    "fibre or thread contamination": "a stray thread or fibre lying across the surface",
    "print or coating defect": "printing or coating that is smeared, scratched through, "
                             "faded or misregistered",
    "stain or discolouration": "a localized patch where the colour has changed against the "
                             "material around it",
    "mould or rot":          "fungal growth or rotting on organic material -- a fuzzy or "
                             "darkened patch spreading across the surface",
    "deformation":           "warped, bent or misshapen geometry left by a moulding or "
                             "handling fault",
}

# Qualitative only. Numeric sizes are banned in the ask -- generators handle "tiny" more
# reliably than "3 mm", and a millimetre figure is meaningless without knowing the scale.
SIZE_WORDS = ["tiny", "very small", "small"]

# Frozen. Nothing here names a dataset or a category -- every category-specific string
# arrives from the VLM JSON.
# Two framings were tried and both failed in an instructive way.
#
#   "Generate the same image ... Keep everything else unchanged."
#       -> the preservation clause dominated and the generator returned the input untouched.
#
#   "Generate the same image in which X shows a defect"
#       -> "the same image" frames the job as copy-then-overlay, so the generator composited
#          a sprite ON TOP of the picture: a rock lying on the rice, an insect sitting on the
#          jelly. Nothing was damaged; something was pasted.
#
# The wording below asks for a natural anomaly IMAGE rather than an edit of an existing one,
# and then says outright that the defect belongs to the material. Preservation is still
# enforced -- by HARD_RULES 1 and 2, where it does not compete with the defect instruction.
TEMPLATE = (
    "Generate a natural-looking anomaly image of this scene: {target} has a {size} "
    "{defect}, {where}. "
    "The defect is part of the material, not laid on top -- its edges, depth and shadow "
    "follow the surface. "
    "Small, clearly visible, photorealistic."
)

HARD_RULES = """HARD RULES YOU MUST FOLLOW

1. Exact Image Preservation: Preserve the original image, geometry, composition, background, texture, lighting, shadows, reflections, perspective, and pixel correspondence everywhere except the requested defect.
2. Photometric & Dimensional Invariance: Preserve color, saturation, exposure, contrast, illumination, material appearance, image dimensions, aspect ratio, crop, framing, and resolution exactly; no resizing, cropping, padding, zooming, or global enhancement.
3. Single Localized Defect: Add exactly ONE small, category-valid, physically realistic anomaly; no reconstruction, recomposition, secondary defects, or any unrequested modification."""


# ---------------------------------------------------------------- VLM-1 wording (editable)
SYS_VLM1 = (
    "You are an industrial quality-inspection expert. Given a defect-free product image, "
    "you name one small realistic defect that could occur on it. You answer only with JSON."
)

ASK_VLM1 = """{dataset_context}

Look at this defect-free industrial inspection image.

  the object is: {object}

Describe ONE small, realistic defect of the following family that could appear on this exact
object during manufacturing or handling:

  family: {family} -- {family_hint}

Use what you can actually see: the material, the surface, the lighting, and how many items
are in frame. Refer to the object as described above, never as something else.
{constraints}{avoid}
Rules:
- If several items are in frame, "target" must say WHICH one, by position.
- "defect" is a short noun phrase of that family. It may add material, remove material, or
  alter something already present.
- It must be SMALL but UNMISTAKABLE -- a definite break in the surface with visible edges,
  not a faint tint or a subtle texture change. A scratch means the surface is torn open, not
  lightly shaded.
- NEVER propose something that is NORMAL for this product. Bubbles, reflections, specular
  highlights, shadows, natural grain or shell texture, print-pattern variation, and the
  ordinary arrangement or count of items are all normal and are NOT defects.
- "defect" must name a specific physical fault -- what happened to the material. "dark spot",
  "mark", "irregularity", "blemish", "anomaly", "imperfection" and "damage" are too vague and
  will be rejected. Say what it is: a torn thread, a bored hole, a shard of glass, a smear of
  ink.
- Phrase "defect" so it is PART OF the material, not resting on it. Say how it sits in the
  surface: "wedged between the grains", "pressed into the set jelly", "torn open across the
  weave", "gouged into the metal". Avoid wording that reads as an object simply placed on
  top.
- Anything foreign must be something that plausibly reaches this product on a production
  line: glass, plastic, metal, grit, thread or fibre, and on food or liquid products a small
  insect. Never leaves, twigs, or outdoor debris.
- Keep it local. Never whole-object damage, never a change to framing or lighting.
- Plain words. No sizes in numbers or millimetres, no metaphors, no optics terms like
  refract, specular, or diffuse.

The three fields are read into this sentence, so they must fit it grammatically:
  "... in which TARGET shows a {size} DEFECT, WHERE."

Reply with only JSON:
{{
  "defect_name": "<2-3 words, database key, e.g. 'shell crack', 'glass shard'>",
  "target":      "<what carries it, with article, e.g. 'the walnut left of centre'>",
  "defect":      "<short noun phrase, e.g. 'crack splitting the shell open', 'torn strip of label'>",
  "where":       "<short position phrase, e.g. 'near its upper seam'>"
}}"""


# ---------------------------------------------------------------- reply validation (editable)
# Plain word lists rather than regexes, so they can be edited without knowing regex syntax.
# They are compiled in the VLM-1 cell.
FIELDS = ("defect_name", "target", "defect", "where")

# Words that describe a SHAPE or a JUDGEMENT rather than a physical fault. A generator given
# "a small dark spot" has nothing to render; given "a bored hole" it does. Always rejected.
VAGUE_WORDS = ["spot", "mark", "blemish", "anomaly", "imperfection", "irregularity",
               "defect", "damage", "flaw", "issue", "artifact", "artefact"]

# Vague in general, legitimate for the stain family only.
STAIN_ONLY_WORDS = ["discolouration", "discoloration", "patch", "area", "region"]
STAIN_FAMILY     = "stain or discolouration"

# NORMAL variation in this data, never a defect. Bubbles in a vial, specular bands on
# dark-field metal and walnut shell grain are the trap MVTec AD 2 is built around.
NORMAL_WORDS = ["bubble", "bubbles", "reflection", "reflections", "highlight", "highlights",
                "glare", "shadow", "shadows", "grain pattern", "natural grain",
                "texture variation", "pattern variation", "lighting"]

# Ignored when checking that "target" names the object -- generic scene words, not nouns.
STOP_WORDS = ["with", "their", "plain", "background", "inside", "several", "single", "loose",
              "flat", "sheet", "pile", "filling", "frame", "lying", "and", "the", "its",
              "suspended", "machined", "whole", "shells", "strips", "bulk", "grains"]

# ---------------------------------------------------------------- generation (editable)
MAX_NEW_TOKENS = 200    # the JSON reply is short; this is headroom
TRIES          = 4      # attempts per image before the fallback fires
TEMP_GREEDY    = 0.0    # first attempt: deterministic
TEMP_SAMPLE    = 0.8    # retries and later images: sampled, so they diverge

CATS = list(CONFIG)
print(f"{len(CATS)} categories configured | source={DEFECT_SOURCE} | "
      f"{N_IMAGES} images per category -> {len(CATS) * N_IMAGES} prompts")

## 3. Defect dictionary and prompt template

Builds the fixed vocabulary of defect families and the sentence they are substituted into.

Families are assigned round-robin from a shuffled list, so a category with three images receives
three different kinds of defect. The vocabulary follows MVTec AD 2 Table 3: print defects,
scratches, cuts, holes, colour inconsistency, loose threads and extra fabric, foreign
contamination, semi-transparent plastic contamination, cracks, missing parts, broken pieces, and
damaged or missing QR codes.

The template names no dataset and no category, so it transfers to other data unchanged.

In [ ]:
def _slug(s):
    """'shell crack' -> 'shell_crack'. Used for filenames and defect-bank keys."""
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")[:32]


def build_paragraph(spec):
    """VLM-1 JSON -> the text handed to the image generator.

    Pure string substitution: no model involved, so the same spec always gives the same
    prompt.
    """
    body = TEMPLATE.format(target=spec["target"], size=spec["size"],
                           defect=spec["defect"], where=spec["where"].rstrip("."))
    return body + "\n\n" + HARD_RULES

## 4. Locate the mounted model and dataset

Resolves the two mount paths. `MODEL_ID` and `DATA_ROOT_ID` from section 2 are used when they
exist; otherwise the notebook searches `/kaggle/input` and prints what it found.

If either line reports a path you did not expect, stop and correct the config before running on.

In [ ]:
from PIL import Image

random.seed(SEED)

# Pin exact normals per category once the bank is settled, e.g.
#   PINNED = {"rice": ["/kaggle/input/<ds>/rice/train/good/000_regular.png", ...]}
PINNED = {}


def find_model():
    """Use MODEL_ID if it exists on disk, else search /kaggle/input for the weights."""
    if MODEL_ID and os.path.isdir(MODEL_ID):
        return MODEL_ID
    if MODEL_ID:
        print(f"  MODEL_ID not on disk ({MODEL_ID}) -- searching instead")
    hits = []
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        if not glob.glob(os.path.join(d, "*.safetensors")):
            continue                                   # config without weights -> skip
        try:
            arch = json.load(open(cfg)).get("architectures", [""])[0]
        except Exception:
            arch = ""
        if "Qwen2_5_VL" in arch or "qwen" in d.lower():
            hits.append((d, arch))
    if not hits:
        raise FileNotFoundError("no mounted Qwen model under /kaggle/input -- attach it, "
                                "or set MODEL_ID to a hub id")
    hits.sort(key=lambda t: t[0])
    for d, a in hits:
        print("  model candidate:", d, "|", a)
    return hits[-1][0]


def find_data_root(max_depth=8):
    """Walk /kaggle/input for the directory holding the most category folders.

    os.walk rather than a fixed glob depth: Kaggle nests mounts differently depending on
    whether something came in as a Dataset or a Model (e.g. models land under
    /kaggle/input/models/<owner>/<name>/<framework>/<variation>/<version>).
    Weight directories are pruned so the walk does not descend into model shards.
    """
    # If DATA_ROOT_ID exists, use it -- but still descend, because reduced copies often add
    # one wrapper folder between the mount point and the category folders.
    root = DATA_ROOT_ID if (DATA_ROOT_ID and os.path.isdir(DATA_ROOT_ID)) else "/kaggle/input"
    if DATA_ROOT_ID and not os.path.isdir(DATA_ROOT_ID):
        print(f"  DATA_ROOT_ID not on disk ({DATA_ROOT_ID}) -- searching /kaggle/input instead")
    best, best_hit = None, 0
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) > max_depth:
            dirnames[:] = []                      # too deep, stop descending
            continue
        if any(f.endswith(".safetensors") or f.endswith(".bin") for f in filenames):
            dirnames[:] = []                      # this is a weights folder, not data
            continue
        hit = sum(d in CATS for d in dirnames)
        if hit > best_hit:
            best, best_hit = dirpath, hit
    if not best:
        print("  nothing matched. what IS mounted under /kaggle/input:")
        for d in sorted(glob.glob("/kaggle/input/*")):
            print("   ", d)
            for sub in sorted(glob.glob(os.path.join(d, "*")))[:8]:
                print("      ", os.path.basename(sub))
        raise FileNotFoundError(
            "no dataset root under /kaggle/input -- attach the dataset, or check that its "
            "category folder names match the keys in CONFIG")
    print(f"  data root: {best}  ({best_hit}/{len(CATS)} categories present)")
    return best


def normals_for(cat, root, k, rng):
    """k defect-free images for a category, sampled with the run seed.

    PINNED overrides sampling entirely -- set it once the bank is settled so the same
    images are used on every rerun.
    """
    if cat in PINNED:
        return PINNED[cat][:k]
    for sub in ("train/good", "validation/good", "test_public/good", "train", "good", ""):
        base = os.path.join(root, cat, sub) if sub else os.path.join(root, cat)
        hits = sorted(glob.glob(os.path.join(base, "*.png")) +
                      glob.glob(os.path.join(base, "*.jpg")))
        if hits:
            return sorted(rng.sample(hits, k=min(k, len(hits))))
    raise FileNotFoundError(f"no normal image for {cat} under {root}")


MODEL_PATH = find_model()
DATA_ROOT  = find_data_root()
CATS       = [c for c in CATS if os.path.isdir(os.path.join(DATA_ROOT, c))]

# Separate RNG stream for image choice, so changing N_IMAGES does not reshuffle families.
_pick_rng = random.Random(SEED)
NORMALS   = {c: normals_for(c, DATA_ROOT, N_IMAGES, _pick_rng) for c in CATS}

print("\nmodel:", MODEL_PATH)
print("\nnormal images sampled per category (each one gets its own prompt):")
for c, paths in NORMALS.items():
    for pth in paths:
        w, h = Image.open(pth).size
        print(f"  {c:12s} {w:5d}x{h:<5d}  {os.path.basename(pth)}")
n_prompts = sum(len(v) for v in NORMALS.values())
print(f"\n{len(CATS)} categories, {n_prompts} images -> {n_prompts} prompts to build")

## 5. Load the model

Loads Qwen2.5-VL in fp16 with `device_map="auto"`, which splits the layers across both T4s. The
device map is printed so you can confirm the split.

`min_pixels` and `max_pixels` cap the visual token count. MVTec AD 2 images are around 2448x2048,
and at native resolution generation is very slow. Categories with extreme aspect ratios can
override the cap in their config entry.

If the processor fails to load, the cell falls through several routes and finally builds it from
its parts. This handles mounts that are missing `preprocessor_config.json`.

In [ ]:
from transformers import AutoProcessor

# transformers v5 renamed the model classes' import path in places; fall back to the generic
# image-text-to-text auto class, which resolves Qwen2.5-VL from the checkpoint config.
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as _VLModel
except ImportError:
    from transformers import AutoModelForImageTextToText as _VLModel

torch.manual_seed(SEED)

_kw = dict(attn_implementation="sdpa",     # Turing: no flash-attn
           device_map="auto")              # split the 7B across both T4s
if LOAD_4BIT:
    # Optional single-card path. Needs bitsandbytes, therefore needs internet.
    from transformers import BitsAndBytesConfig
    _kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

_t0 = time.time()
# v5 renamed torch_dtype -> dtype. Try the new name, fall back to the old one.
try:
    model = _VLModel.from_pretrained(MODEL_PATH, dtype=torch.float16, **_kw).eval()
except TypeError:
    model = _VLModel.from_pretrained(MODEL_PATH, torch_dtype=torch.float16, **_kw).eval()


# ---------------------------------------------------------------- processor
# Kaggle model mounts are sometimes missing preprocessor_config.json, in which case
# AutoProcessor cannot resolve the image processor and raises
#   "Unrecognized image processor in <path>"
# even though the weights are fine. The cascade below tries the normal routes first, then
# builds the processor from its parts.
print("files in model dir:", sorted(os.listdir(MODEL_PATH)))


def _patched_config_dir(path, dst="/kaggle/working/_qwen_processor"):
    """Copy only the small config/tokenizer files to a writable dir and inject the keys
    transformers v5 needs.

    This checkpoint was saved by transformers 4.x, whose preprocessor_config.json has no
    `image_processor_type`. v5 no longer falls back to config.json's `model_type`, so
    AutoProcessor cannot resolve the class. Adding the key by hand fixes it. The mount is
    read-only, hence the copy -- weights are NOT copied, only json/txt (a few MB).
    """
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(path):
        if f.endswith((".json", ".txt")) and f != "model.safetensors.index.json":
            shutil.copy2(os.path.join(path, f), os.path.join(dst, f))

    pc = os.path.join(dst, "preprocessor_config.json")
    cfg = json.load(open(pc)) if os.path.exists(pc) else {}
    cfg.setdefault("image_processor_type", "Qwen2_5_VLImageProcessor")
    cfg.setdefault("processor_class", "Qwen2_5_VLProcessor")
    cfg.setdefault("video_processor_type", "Qwen2_5_VLVideoProcessor")
    json.dump(cfg, open(pc, "w"), indent=2)
    return dst


def load_processor(path):
    errors = []

    for kw in (dict(min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, use_fast=False),
               dict(use_fast=False)):                    # 1) straight AutoProcessor
        try:
            return AutoProcessor.from_pretrained(path, **kw)
        except Exception as e:
            errors.append(f"AutoProcessor({', '.join(kw)}): {type(e).__name__}: {e}")

    try:                                                 # 2) patched config dir
        pdir = _patched_config_dir(path)
        proc = AutoProcessor.from_pretrained(pdir, min_pixels=MIN_PIXELS,
                                             max_pixels=MAX_PIXELS, use_fast=False)
        print(f"  loaded via patched config dir ({pdir}) -- "
              "checkpoint predates the v5 image_processor_type key")
        return proc
    except Exception as e:
        errors.append(f"patched dir: {type(e).__name__}: {e}")

    try:                                                 # 3) concrete class, patched dir
        from transformers import Qwen2_5_VLProcessor
        return Qwen2_5_VLProcessor.from_pretrained(_patched_config_dir(path))
    except Exception as e:
        errors.append(f"Qwen2_5_VLProcessor: {type(e).__name__}: {e}")

    try:                                                 # 4) assemble from parts
        from transformers import AutoTokenizer, Qwen2_5_VLProcessor
        try:
            from transformers import Qwen2_5_VLImageProcessor as _IP
        except ImportError:
            from transformers import Qwen2VLImageProcessor as _IP
        # v5 requires a video processor even for image-only use.
        _VP = None
        for _n in ("Qwen2_5_VLVideoProcessor", "Qwen2VLVideoProcessor"):
            try:
                _VP = getattr(__import__("transformers", fromlist=[_n]), _n)
                break
            except Exception:
                pass
        tok  = AutoTokenizer.from_pretrained(path)
        ip   = _IP(min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
        kw   = dict(image_processor=ip, tokenizer=tok)
        if _VP is not None:
            kw["video_processor"] = _VP()
        print("  built processor from tokenizer + image processor")
        return Qwen2_5_VLProcessor(**kw)
    except Exception as e:
        errors.append(f"manual build: {type(e).__name__}: {e}")

    raise RuntimeError("could not build a processor:\n  " + "\n  ".join(errors))


processor = load_processor(MODEL_PATH)

# Apply the pixel caps if they did not go in through the constructor.
_ip = getattr(processor, "image_processor", None)
if _ip is not None:
    for _k, _v in (("min_pixels", MIN_PIXELS), ("max_pixels", MAX_PIXELS)):
        try:
            setattr(_ip, _k, _v)
        except Exception:
            pass

# A hand-built processor has no chat template; recover it from the mount or the tokenizer,
# otherwise apply_chat_template later would fail.
if not getattr(processor, "chat_template", None):
    _ct = None
    _ctf = os.path.join(MODEL_PATH, "chat_template.json")
    if os.path.exists(_ctf):
        _ct = json.load(open(_ctf)).get("chat_template")
    if _ct is None:
        _ct = getattr(getattr(processor, "tokenizer", None), "chat_template", None)
    if _ct:
        processor.chat_template = _ct
        print("  chat template recovered")
    else:
        print("  WARNING: no chat template found -- apply_chat_template may fail")

LOAD_SECONDS = time.time() - _t0
MODEL_REV = os.path.basename(MODEL_PATH.rstrip("/"))

print(f"loaded in {LOAD_SECONDS:.0f}s | 4bit={LOAD_4BIT}")
print("device map:", getattr(model, "hf_device_map", "single device"))


def _set_pixel_budget(max_pixels):
    """Swap the image processor's area budget for one call, returning the previous value.

    Categories with extreme aspect ratios (sheet_metal is 4:1) lose too much detail under
    the shared budget, because the cap is on total area and shrinks both axes.
    """
    ip = getattr(processor, "image_processor", None)
    if ip is None:
        return None
    prev = getattr(ip, "max_pixels", None)
    try:
        ip.max_pixels = max_pixels
        # Some versions mirror the caps inside a `size` dict; keep it consistent.
        if isinstance(getattr(ip, "size", None), dict) and "longest_edge" in ip.size:
            ip.size["longest_edge"] = max_pixels
    except Exception:
        return None
    return prev


def vlm(img_path, system, question, max_new_tokens=200, temperature=0.0, max_pixels=None):
    """One VLM call: image + question -> (raw text reply, generate seconds).

    The timer starts only AFTER the image has been read, preprocessed and moved to the GPU,
    so the reported figure is the time Qwen spends looking at the image and answering --
    not disk I/O or tokenisation.

    temperature=0 is greedy and deterministic; >0 samples, which is how later images in a
    category are pushed away from the first.
    """
    prev_budget = _set_pixel_budget(max_pixels) if max_pixels else None
    img  = Image.open(img_path).convert("RGB")
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": question}]}]
    text   = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(model.device)
    if prev_budget is not None:
        _set_pixel_budget(prev_budget)          # restore before the next category
    kw = dict(max_new_tokens=max_new_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw.update(temperature=temperature, top_p=0.9)

    if torch.cuda.is_available():
        torch.cuda.synchronize()          # do not time work still queued from before
    t0 = time.time()                      # <-- clock starts: image is now with Qwen
    with torch.inference_mode():
        out = model.generate(**inputs, **kw)
    if torch.cuda.is_available():
        torch.cuda.synchronize()          # generate is async; wait before stopping the clock
    secs = time.time() - t0

    # Strip the prompt tokens; keep only what the model generated.
    reply = processor.decode(out[0][inputs.input_ids.shape[1]:],
                             skip_special_tokens=True).strip()
    return reply, secs


def grab_json(raw):
    """Pull the first JSON object out of a reply, tolerating stray prose or code fences."""
    m = re.search(r"[\[{].*[\]}]", raw, re.S)
    if not m:
        raise ValueError("no JSON in reply")
    return json.loads(m.group(0))

## 6. The question put to the model

Defines the system prompt and the question. Neither mentions a specific category.

The model is asked for four short fields as JSON: which item carries the defect, what the defect
is, where it sits, and a name for it. An unparseable reply, or one repeating a defect already used
in that category, is retried up to three times and then falls back to a config default.

Edit `SYS_VLM1` and `ASK_VLM1` in section 2 if you want different wording.

In [ ]:
# Compiled from the editable word lists in the config cell.
_rx = lambda words: re.compile(r"\b(" + "|".join(words) + r")\b", re.I)
VAGUE              = _rx(VAGUE_WORDS)
VAGUE_UNLESS_STAIN = _rx(STAIN_ONLY_WORDS)
NORMAL             = _rx(NORMAL_WORDS)


def validate(spec, cat, family, taken):
    """Raise if the reply is unusable. Every raise triggers a retry with feedback."""
    missing = [f for f in FIELDS if not spec.get(f)]
    if missing:
        raise ValueError("missing fields: " + ", ".join(missing))
    if spec["defect_name"] in taken:
        raise ValueError("duplicate defect_name: " + spec["defect_name"])

    d = spec["defect"]

    # Under DEFECT_SOURCE == "config" the VLM may not invent a defect: what it returns has to
    # correspond to one of the entries listed for this category.
    if DEFECT_SOURCE == "config":
        allowed_defects = CONFIG.get(cat, {}).get("defects", [])
        if allowed_defects:
            words = set(re.findall(r"[a-z]{4,}", d.lower()))
            hit = any(words & set(re.findall(r"[a-z]{4,}", a.lower()))
                      for a in allowed_defects)
            if not hit:
                raise ValueError(f"{d!r} is not one of the listed defects for {cat}")

    if NORMAL.search(d) or NORMAL.search(spec.get("defect_name", "")):
        raise ValueError(f"names normal variation, not a defect: {d!r}")
    if VAGUE.search(d):
        raise ValueError(f"too vague, name the physical fault: {d!r}")
    if family != STAIN_FAMILY and VAGUE_UNLESS_STAIN.search(d):
        raise ValueError(f"too vague for this family, name the physical fault: {d!r}")
    if len(d.split()) < 2:
        raise ValueError(f"defect is one word, needs a noun phrase: {d!r}")

    # The object must be called what it is -- "the drink" for a tub of jelly is a mis-read.
    # The object must be called what it is. "the drink" for a tub of jelly is a mis-read and
    # every prompt built from it would be wrong. Matching is on 3+ char stems so "can" and
    # "vial" survive, and plurals match ("walnut" ~ "walnuts"). The category name is folded in
    # so "the wall plug at the left" matches the wallplugs category.
    stop = set(STOP_WORDS)

    def stems(txt):
        return {w[:5] for w in re.findall(r"[a-z]{3,}", txt.lower()) if w not in stop}

    allowed = stems(CONFIG.get(cat, {}).get("object", "")) | stems(cat.replace("_", " "))
    if allowed and not (stems(spec["target"]) & allowed):
        raise ValueError(f"target {spec['target']!r} does not name the object "
                         f"({CONFIG.get(cat, {}).get('object', cat).split(',')[0]})")
    return spec


def _fallback(cat, family, size, last_raw):
    """Used only after every retry failed. Built from CONFIG so it is still category-valid,
    and it keeps the last raw reply so the failure can be diagnosed from the JSON."""
    cfg = CONFIG.get(cat, {})
    defect = (cfg.get("defects") or ["surface fault"])[0]
    target = "the " + (cfg.get("object", "object").split(" with ")[0]
                       .replace("a printed ", "").replace("a sealed ", "")
                       .replace("a single sealed ", "").replace("a flat sheet of ", "")
                       .replace("a loose pile of ", "").replace("several ", "")
                       .replace("loose ", "").split(",")[0].split(" lying")[0]
                       .split(" filling")[0].strip())
    return {"defect_name": defect.split()[0] + " fault", "target": target,
            "defect": defect, "where": "on its visible surface",
            "size": size, "family": family, "FALLBACK": True,
            "last_raw_reply": last_raw}


def _constraints(cat):
    """Config hints injected into the ask. Empty when the config has nothing to say."""
    cfg, out = CONFIG.get(cat, {}), ""
    if DEFECT_SOURCE != "derive" and cfg.get("defects"):
        out += "The defect must be one of: " + ", ".join(cfg["defects"]) + ".\n"
    if cfg.get("notes"):
        out += "Note: " + cfg["notes"] + ".\n"
    return out


def vlm1(cat, img, family, size, taken=(), temperature=None, tries=None):
    """One Anomaly Prompt spec. Returns (spec, raw_reply, retries, generate seconds).

    A rejected reply is retried WITH the reason appended, so the model is told what was
    wrong rather than being asked the same question again. Sampling is turned on from the
    second attempt so a greedy failure does not simply repeat.
    """
    tries = TRIES if tries is None else tries
    temperature = TEMP_GREEDY if temperature is None else temperature
    mp    = CONFIG.get(cat, {}).get("max_pixels", MAX_PIXELS)
    obj   = CONFIG.get(cat, {}).get("object", "an industrial part")
    avoid = ("Do NOT reuse any of these defect names: " + ", ".join(taken) + ".\n") if taken else ""
    base  = ASK_VLM1.format(dataset_context=DATASET_CONTEXT, object=obj, family=family,
                            family_hint=FAMILIES[family], size=size,
                            constraints=_constraints(cat), avoid=avoid)
    gen_secs, raw, why = 0.0, "", None
    for t in range(tries):
        q = base if why is None else (
            base + f"\n\nYour previous answer was rejected: {why}. Fix exactly that and "
                   f"answer again, JSON only.")
        raw, secs = vlm(img, SYS_VLM1, q, max_new_tokens=MAX_NEW_TOKENS,
                        temperature=temperature if t == 0 else TEMP_SAMPLE, max_pixels=mp)
        gen_secs += secs
        try:
            spec = validate(grab_json(raw), cat, family, taken)
            spec["size"], spec["family"] = size, family
            return spec, raw, t, gen_secs
        except Exception as e:
            why = f"{type(e).__name__}: {e}"
            print(f"      retry {t + 1}/{tries} -- {why}")
    print(f"      !! FALLBACK for {cat} / {family} -- last reply: {raw[:120]!r}")
    return _fallback(cat, family, size, raw), raw, tries, gen_secs

## 7. Run

For each category: sample `N_IMAGES` normals, assign each a different defect family, call the
model once per image, write the results.

Runtime is roughly a few seconds per image. Progress is printed per image with the defect name and
the generate time.

In [ ]:
rng = random.Random(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

manifest, rows = {}, []
run_t0 = time.time()

for cat in CATS:
    cat_dir = os.path.join(OUT_DIR, cat)
    os.makedirs(cat_dir, exist_ok=True)
    paths = NORMALS[cat]

    # One distinct family per image -- this is what stops a category's prompts converging.
    # Only families that can physically occur on this category. Without this you get
    # "walnut print defect" or "can missing part" -- the family pool is uniform otherwise.
    pool = CONFIG.get(cat, {}).get("families") or list(FAMILIES)
    fams = FAMILY_PLAN.get(cat) or rng.sample(pool, k=min(len(paths), len(pool)))

    print(f"[{cat}]")
    taken = []

    for i, (img, family) in enumerate(zip(paths, fams)):
        # Copy the image in; its filename is the tag used everywhere downstream.
        tag = f"Normal_{cat}_{os.path.basename(img)}"
        shutil.copy2(img, os.path.join(cat_dir, tag))

        size = rng.choice(SIZE_WORDS)
        # First image greedy for a stable anchor; later ones sampled to diverge.
        spec, raw, retries, secs = vlm1(cat, img, family, size, taken=taken,
                                        temperature=TEMP_GREEDY if i == 0 else TEMP_SAMPLE)
        taken.append(spec["defect_name"])
        para = build_paragraph(spec)

        rec = {"image": tag, "category": cat, "source_path": img,
               "spec": spec, "paragraph": para,
               "provenance": {"model": MODEL_REV, "model_path": MODEL_PATH,
                              "load_4bit": LOAD_4BIT, "seed": SEED,
                              "defect_source": DEFECT_SOURCE, "family": family,
                              "size_word": size, "retries": retries,
                              "gen_seconds": round(secs, 2), "raw_reply": raw}}
        manifest[tag] = rec
        rows.append({"category": cat, "image": tag, "family": family,
                     "defect_name": spec["defect_name"], "size": size,
                     "target": spec["target"], "defect": spec["defect"],
                     "where": spec["where"], "prompt": para,
                     "retries": retries, "gen_seconds": round(secs, 2)})

        print(f"    {tag}")
        print(f"      [{family:<22s}] {spec['defect_name']:<26s} {secs:6.1f}s"
              f"{'  (' + str(retries) + ' retries)' if retries else ''}")
    print()

RUN_SECONDS = time.time() - run_t0

# CSV is the durable artefact -- prompts survive a kernel reset even if nothing else does.
with open(os.path.join(OUT_DIR, "prompts.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

with open(os.path.join(OUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump({"config": {"model": MODEL_REV, "model_path": MODEL_PATH, "seed": SEED,
                          "defect_source": DEFECT_SOURCE, "n_images": N_IMAGES,
                          "categories": CATS, "load_seconds": round(LOAD_SECONDS, 1),
                          "run_seconds": round(RUN_SECONDS, 1)},
               "entries": manifest}, f, indent=2)

print(f"{len(rows)} prompts over {len(CATS)} categories -> {OUT_DIR}")
print(f"  images   : {len(rows)} copied into per-category folders")
print(f"  prompts  : prompts.csv")
print(f"  manifest : manifest.json")

## 8. Timing

Reports how long the model takes per image. The clock starts once the encoded image reaches the
model, so disk read, preprocessing and tokenisation are excluded.

Model load is reported separately because it is paid once per session rather than per image.

In [ ]:
print(f"model load (one-off, excluded below) : {LOAD_SECONDS:7.1f}s\n")
print(f"{'category':<14s}{'images':>8s}{'total s':>10s}{'mean s':>9s}"
      f"{'min s':>8s}{'max s':>8s}{'retries':>9s}")
print("-" * 66)

tot_s = tot_n = tot_r = 0
for cat in CATS:
    rs = [r for r in rows if r["category"] == cat]
    if not rs:
        continue
    ss = [r["gen_seconds"] for r in rs]
    r_ = sum(r["retries"] for r in rs)
    tot_s, tot_n, tot_r = tot_s + sum(ss), tot_n + len(rs), tot_r + r_
    print(f"{cat:<14s}{len(rs):>8d}{sum(ss):>10.1f}{sum(ss)/len(ss):>9.1f}"
          f"{min(ss):>8.1f}{max(ss):>8.1f}{r_:>9d}")

print("-" * 66)
print(f"{'ALL':<14s}{tot_n:>8d}{tot_s:>10.1f}{tot_s/max(tot_n,1):>9.1f}"
      f"{'':>8s}{'':>8s}{tot_r:>9d}")
print(f"\nmean per image (VLM generate only) : {tot_s/max(tot_n,1):.1f}s")
print(f"wall clock for the whole run       : {RUN_SECONDS:.1f}s")

## 9. Inspect

Shows every normal image beside the prompt built from it.

Check these before generating. The failure worth catching is a prompt that is plausible in the
abstract but impossible in this particular image: a defect placed where the lighting cannot
resolve it, or on an item that is not in frame. Delete the row from `prompts.csv` and re-run this
category if you find one.

In [ ]:
import matplotlib.pyplot as plt
import textwrap

for cat in CATS:
    recs = [r for r in manifest.values() if r["category"] == cat]
    if not recs:
        continue
    print("=" * 100)
    print(f"  {cat.upper()}   ({len(recs)} images)")
    print("=" * 100)

    for r in recs:
        p = r["provenance"]
        fig, ax = plt.subplots(figsize=(6, 4.5))
        ax.imshow(Image.open(r["source_path"]).convert("RGB"))
        ax.set_title(r["image"], fontsize=9)      # image tag == filename on disk
        ax.axis("off")
        plt.tight_layout()
        plt.show()

        print(f"  image  : {r['image']}")
        print(f"  family : {p['family']}   |   defect: {r['spec']['defect_name']}   "
              f"|   {p['gen_seconds']}s"
              f"{'   (' + str(p['retries']) + ' retries)' if p['retries'] else ''}")
        print("  prompt :")
        for line in textwrap.wrap(r["paragraph"].split("\n\nHARD RULES")[0], 92):
            print("           " + line)
        print()

# -----------------------------------------------------------------------------------------
# HANDOFF -- MANUAL AT PRESENT.
# Stage 1 ends here. Every normal image above now has its Anomaly Prompt. The prompt and its
# matching normal image are currently pasted into ChatGPT BY HAND, one pair at a time, and
# the returned anomaly image I_a is saved back for Stage 2 (DiffMask).
#
# Cost of that manual step: roughly 50-60 s per image, so ~24 min for the 24-image bank.
# This is the 60 s/call figure the paper's efficiency analysis assumes.
#
# Nothing in this notebook calls a generator. Moving the handoff onto the OpenAI Images API
# would cut the per-image time substantially -- no browser round trip, no copy-paste, and
# calls can run concurrently -- as well as recording the model version, which the manual
# route cannot. That module reads manifest.json and writes one I_a per row.
# -----------------------------------------------------------------------------------------

## 10. Next step

Each normal image now has one anomaly prompt in `prompts.csv` and `manifest.json`.

Pass each prompt with its normal image to an image generation model and save the returned anomaly
image alongside the normal one, named so the two pair up:

```
<category>/<id>_regular.png     the normal image
<category>/<id>_anomaly.png     the generated anomaly
```

That layout is what Module 2 expects. Generation currently costs roughly 50 to 60 seconds per
image when done by hand; an API call is faster and records the model version against each image.

This cost is paid once per category. Module 3 replays the banked defects onto new hosts without
calling a generator again.